# 01 — Dataset Audit (mandatory before training)

**What this notebook does**: indexes every observation in `NASA_EyeTracking_Dataset`, checks integrity, timing,
image quality and label coverage, writes the train / validation / test manifests, and gives a **PASS / FAIL**
training-readiness decision. Later notebooks refuse to train unless this decision is PASS.

**What it never does**: modify, move or copy images; train anything; use the test set for any decision.
Checking that test files exist and decode is integrity checking, not model selection.

**Run time**: `DECODE_MODE = "sample"` takes minutes; `"full"` decodes all ~200,000 PNGs (tens of minutes,
CPU only). Run `"full"` once before you train.

**Outputs** (all inside `training_workspace/`):
`reports/dataset_audit.html`, `reports/dataset_audit.json`, `reports/exclusion_report.csv`,
`manifests/master_manifest.parquet`, `manifests/{train,validation,test}.parquet`, `figures/audit/*`.

In [ ]:
# --- Workspace setup -----------------------------------------------------------
# WHAT: makes the `src` package importable and loads training_config.yaml.
# WHY : every notebook uses the same code and the same configuration.
# PATHS: resolved from this notebook's folder, never from the terminal's cwd.
import sys
from pathlib import Path

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
if not (WORKSPACE / "src" / "config.py").is_file():
    raise RuntimeError(f"Open this notebook from training_workspace/notebooks (cwd = {Path.cwd()})")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.config import load_config
cfg = load_config(WORKSPACE / "training_config.yaml")
cfg.paths.ensure_output_dirs()
W, H = cfg.screen_px
print("Workspace     :", cfg.paths.workspace)
print("Dataset root  :", cfg.paths.dataset_root, "(read-only)")
print("Screen        :", W, "x", H, "px")

## 1. Settings for this audit
`DECODE_MODE` controls how many images are opened:
* `"none"` – metadata and file existence only (seconds).
* `"sample"` – `SAMPLE_TRIALS_PER_SESSION` random trials from **every** session (minutes).
* `"full"` – every image (recommended once before training).

`VERIFY_RAW_HASH = True` additionally recomputes the SHA-256 of every decoded raw frame and compares it with the
hash stored at capture time (proves the original RGB pixels are unchanged; slower).

In [ ]:
DECODE_MODE = "sample"            # "none" | "sample" | "full"
SAMPLE_TRIALS_PER_SESSION = 10
VERIFY_RAW_HASH = False
THREADS = 8                       # image decoding threads (I/O-bound; safe on Windows)

## 2. Discover sessions and count observations (fast, metadata only)
The dataset is searched **recursively** for `annotations.jsonl`, so sessions nested at any depth are found.
Pilot sessions (under `pilot/`) are recognised and **excluded** from every split.

**Important distinction**
* *Image files* — 4 per observation (`raw`, `face`, `left_eye`, `right_eye`): four views of ONE frame.
* *Observations* — one labelled camera frame (one annotation row).
* *Fixation trials* — 10 consecutive frames looking at the same target. Frames of a trial are near-duplicates,
  so the number of **independent gaze positions** is the number of trials, not frames.

In [ ]:
from src.dataset import build_master_manifest, effective_sample_report

master, malformed, session_infos, pilot_dirs = build_master_manifest(cfg)
print(f"observations (labelled frames): {len(master):,}")
print(f"image files referenced        : {4 * len(master):,}  (4 per observation)")
print(f"fixation trials               : {master['trial_id'].nunique():,}")
print(f"sessions                      : {master['session_dir'].nunique()}")
print(f"malformed annotation lines    : {len(malformed)}")
print(f"pilot sessions ignored        : {len(pilot_dirs)}")
effective_sample_report(master).assign(split=lambda d: d.participant_id.map(cfg.split_of))

## 3. Run all integrity, timing, quality and coverage checks
Checks performed (see `src/audit.py` for each implementation):

| Group | Checks |
|---|---|
| Integrity | file exists & non-empty; decodes; crop sizes (224 / 112); 3 channels; raw size = metadata; optional raw pixel hash |
| Labels | coordinates finite and on-screen; px ↔ normalised consistent; label = `targets.json`; screen size = participant monitor |
| IDs | participant/session match their folders; duplicate sample IDs; identical raw frames (by stored hash) |
| Splits | recorded split = documented split; every participant in exactly one split |
| Timing | frame captured ≥ 0.8 s after target onset (not during the transition); slow trials flagged; frames inside paused intervals (from the capture app's event logs); state = CAPTURING |
| Quality | brightness, contrast, blur, over/under-exposure, near-duplicate rate, adjacent-frame correlation, blink heuristic, left/right eye ordering, head-position spread |
| Coverage | centre / edge / corner share, 8×8 grid balance, repeated locations |

**Exclusion policy**: only *critical* row problems exclude a frame, and then the whole trial is excluded so no
partial trial remains. Everything excluded is listed with a reason in `exclusion_report.csv`. Warnings (slow trial,
possible blink, dark images) are kept and flagged — nothing is removed silently.

In [ ]:
from src.audit import run_audit, make_audit_figures

audit = run_audit(cfg, decode_mode=DECODE_MODE, sample_trials_per_session=SAMPLE_TRIALS_PER_SESSION,
                  verify_raw_hash=VERIFY_RAW_HASH, max_threads=THREADS)
print("Decision:", audit.decision)

## 4. Findings — critical errors, warnings and acceptable variations

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)
findings = pd.DataFrame(audit.findings)
order = {"CRITICAL": 0, "WARNING": 1, "ACCEPTABLE": 2}
findings.sort_values("level", key=lambda s: s.map(order))

## 5. Exclusion report
One row per excluded observation and reason. An empty table means nothing was excluded.
**How to read it**: `same_trial_as_excluded_frame` means another frame of the same trial had a critical problem.

In [ ]:
ex = audit.exclusions
print(f"excluded observations: {ex['sample_id'].nunique() if len(ex) else 0}")
ex["reason"].value_counts() if len(ex) else ex

## 6. Image quality and head-position coverage
Values from the capture app (all frames) and from decoded images (sampled or full).
* *adjacent_frame_dhash_hamming* near 0 and *pixel correlation* near 1 confirm that consecutive frames of a trial
  are near-duplicates — why sampling and statistics are organised by trial.
* The blink heuristic is a screening aid, not a validated detector; flagged frames are kept.

In [ ]:
import json
print(json.dumps(audit.quality, indent=2, default=str)[:6000])

## 7. Label coverage on the participant monitor
**How to interpret**: targets stop ~5 % from the screen edge (capture margin). The model therefore has little
evidence at the extreme edges and corners; expect larger errors there and keep interactive elements away from the
outer 5 %.

In [ ]:
print(json.dumps(audit.coverage, indent=2))

## 8. Figures (saved as PNG + SVG under `figures/audit/`)

In [ ]:
from IPython.display import Image, display
figs = make_audit_figures(audit, cfg, cfg.paths.figures / "audit")
for name, path in figs.items():
    print(name, "->", path)
    display(Image(filename=str(path)))

## 9. Write the reports and the split manifests
Manifests store **paths relative to the dataset root**; no image is copied. `write_split_manifests`
re-asserts subject disjointness and also checks that no sample ID, trial or identical raw frame appears in two
splits (hash comparison — no pairwise image comparison needed).

In [ ]:
from src.audit import write_reports
outputs = write_reports(audit, cfg)
for k, v in outputs.items():
    print(f"{k:22s} {v}")

## 10. Explicit leakage proofs

In [ ]:
from src.dataset import load_split, TestSetSealed
train_df, val_df = load_split(cfg, "train"), load_split(cfg, "validation")
test_ids = set(pd.read_parquet(cfg.paths.manifests / "test.parquet", columns=["participant_id"])["participant_id"])
tr, va = set(train_df.participant_id), set(val_df.participant_id)
assert tr & va == set(), "train ∩ validation subjects must be empty"
assert tr & test_ids == set(), "train ∩ test subjects must be empty"
assert va & test_ids == set(), "validation ∩ test subjects must be empty"
print("train      :", sorted(tr))
print("validation :", sorted(va))
print("test       :", sorted(test_ids), "(sealed)")
try:
    load_split(cfg, "test")
except TestSetSealed as e:
    print("Test split correctly sealed:", e)
info = json.loads((cfg.paths.manifests / "manifest_info.json").read_text())
info["leakage_checks"]

## 11. Training-readiness decision
If this cell raises, read the critical findings above and the HTML report, fix the cause in the data (never by
editing manifests by hand), and rerun this notebook. Notebooks 03–06 call `require_audit_pass()` and stop if the
decision is not PASS.

In [ ]:
from src.dataset import require_audit_pass
summary = require_audit_pass(cfg)
print("PASS - training may proceed.")
print("Open the report in a browser:", cfg.paths.reports / "dataset_audit.html")